<a href="https://colab.research.google.com/github/cbonnin88/EcoFlux/blob/main/EcoFlux_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import plotly.express as px
import plotly.graph_objects as go

# **1. Load the Dataset**

In [ ]:
df_product = pd.read_csv('ecoflux_amplitude_events.csv')

# **2. Feature Engineering**

In [ ]:
# I aggregated data per user to create features for the model

user_features = df_product.groupby('user_id').agg({
    'activity_score':['mean','std','max'],
    'country':'first',
    'subscription_plan':'first',
    'device_type':'first'
})

In [ ]:
user_features.columns = ['avg_score','score_volatility','max_score','country','plan','device']

# **Defining the Target: Is the user a "Power User" (Top 25% of max scores) ?**

In [ ]:
threshold = user_features['max_score'].quantile(0.75)
user_features['is_power_user'] = (user_features['max_score'] > threshold).astype(int)

# **3. Data Preprocessing (Encoding Categorical Variables)**

In [ ]:
# ML models need numbers, so we convert 'country', 'plan', etc...
df_ml = pd.get_dummies(user_features, columns=['country','plan','device'])

X = df_ml.drop(['is_power_user'],axis=1)
y = df_ml['is_power_user']

# **4. Train/Test Split**

In [ ]:
X_train, X_test, y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=42 )

# **5. Train Random Forest**

In [ ]:
model = RandomForestClassifier(n_estimators=100,random_state=42)
model.fit(X_train,y_train)

RandomForestClassifier(random_state=42)

# **6. Model Insights: Feature Importance**

In [ ]:
importances = pd.DataFrame({
    'feature':X.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance',ascending=False)

# **7. Visualize with Plotly**

In [ ]:
fig = px.bar(
    importances.head(10),
    x='importance',
    y='feature',
    orientation='h',
    title='What Drive High Green Engagement? (Feature Importance)',
    labels={'importance':'Predictive Power','feature':'User Attribute'},
    color='importance',
    color_continuous_scale='Viridis'
)

fig.show()

# **The Confusion Matrix**

In [ ]:
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test,y_pred)

In [ ]:
fig_cm = px.imshow(
    cm,
    text_auto=True,
    labels=dict(x='Predicted Label',y='Actual Label'),
    x=['Regular User','Power User'],
    y=['Regular User','Power User '],
    title='Confusion Matrix: Predictivie Accuracy',
    color_continuous_scale = 'Greens'
)

fig_cm.show()

# **Preparing the ML Predictions for Export (Looker Studio)**

In [ ]:
propensity_scores = model.predict_proba(X)[:, 1]

In [ ]:
predictions_df = pd.DataFrame({
    'user_id': df_ml.index,
    'propensity_score': propensity_scores,
    'is_predicted_power_user': model.predict(X)
})

In [ ]:
# I merged back with some metadata for better context in BigQuery
final_export = predictions_df.merge(
    user_features[['country','plan']].reset_index(),
    on='user_id'
)

In [ ]:
final_export.to_csv('ecoflux_ml_predictions.csv', index=False)

In [20]:
fig_dist = px.histogram(
    final_export,
    x="propensity_score",
    color="plan",
    marginal="box",
    title="Distribution of Power User Propensity by Subscription Tier",
    color_discrete_sequence=px.colors.sequential.Viridis,
    labels={'propensity_score': 'Likelihood of becoming a Power User'}
)
fig_dist.update_layout(template="plotly_white")
fig_dist.show()